# 👗 MyFitDaily - AI Virtual Try-On Server (Google Colab T4 Free)
### Hệ thống Thử Đồ Ảo AI Chuẩn Thương Mại & Tự Do Huấn Luyện (Training/Fine-tuning)

Notebook này biến Google Colab (với GPU **NVIDIA Tesla T4 16GB VRAM** hoàn toàn miễn phí) thành một **AI Server riêng của bạn**:
1. Chạy model **CatVTON / IDM-VTON** với tốc độ cực nhanh (~4-6s/lần thử đồ).
2. Tự động mở đường hầm bảo mật (**Cloudflare Tunnel HTTPS**) kết nối thẳng vào Web MyFitDaily.
3. Cung cấp sẵn công cụ chuẩn bị dữ liệu và script **Training / Fine-tune** phong cách thời trang riêng.

👉 **Hướng dẫn:** Chọn menu `Runtime` -> `Change runtime type` -> Chọn **T4 GPU** -> Bấm `Run all` (hoặc `Ctrl + F9`).

In [ ]:
# BƯỚC 1: Kiểm tra GPU NVIDIA T4 do Google cấp
!nvidia-smi

In [ ]:
# BƯỚC 2: Cài đặt thư viện AI & Công cụ tạo Tunnel kết nối Web
import os, sys

# Clone CatVTON mã nguồn mở từ GitHub
if not os.path.exists('/content/CatVTON'):
    !git clone https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON

%cd /content/CatVTON
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn python-multipart

# Cài đặt Cloudflare Tunnel binary trực tiếp
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("✓ Hoàn tất cài đặt môi trường AI!")


In [ ]:
# BƯỚC 3: Tải Model CatVTON vào GPU (Chỉ tốn ~6GB VRAM, siêu mượt trên T4)
import torch
from model.pipeline import CatVTONPipeline

print("Đang tải trọng số model CatVTON vào GPU T4...")
pipeline = CatVTONPipeline(
    base_ckpt="booksforear/sd-image-inpainting",
    attn_ckpt="zhengchong/CatVTON",
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,
    use_tf32=True,
    device="cuda"
)
print("✓ Model CatVTON đã sẵn sàng phục vụ!")

In [ ]:
# BƯỚC 4: Khởi chạy API Server & Mở đường link kết nối với MyFitDaily Web
import io, base64, subprocess, time, re, threading
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI(title="MyFitDaily AI Try-On Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def health():
    return {"status": "online", "model": "CatVTON (Google Colab Tesla T4)"}

@app.post("/tryon")
async def tryon(
    person_image: UploadFile = File(...),
    garment_image: UploadFile = File(...),
    category: str = Form("upper_body")
):
    try:
        person = Image.open(io.BytesIO(await person_image.read())).convert("RGB")
        garment = Image.open(io.BytesIO(await garment_image.read())).convert("RGB")

        # Chạy inference CatVTON
        result = pipeline(
            image=person,
            condition_image=garment,
            num_inference_steps=30,
            guidance_scale=2.5,
            seed=42
        )[0]

        buf = io.BytesIO()
        result.save(buf, format="JPEG", quality=95)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

        return {
            "success": True,
            "image_url": f"data:image/jpeg;base64,{b64}",
            "model": "CatVTON (Private Colab T4)"
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={"success": False, "error": str(e)})

# 1. Khởi chạy Uvicorn trong background thread
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)

# 2. Khởi động Cloudflare Tunnel và bắt đúng URL ngẫu nhiên của người dùng
!pkill -f cloudflared > /dev/null 2>&1 || true
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

tunnel_url = None
print("⏳ Đang thiết lập đường hầm Cloudflare Tunnel bảo mật...")
for _ in range(40):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    # Bỏ qua link API hệ thống của Cloudflare (api.trycloudflare.com), chỉ lấy link tunnel riêng
    if m and 'api.trycloudflare.com' not in m.group(0):
        tunnel_url = m.group(0)
        break

print("=" * 65)
if tunnel_url:
    print("🎉 AI SERVER ĐÃ KHỞI CHẠY THÀNH CÔNG!")
    print(f"👉 COPY LINK NÀY DÁN VÀO WEB MYFITDAILY: {tunnel_url}")
else:
    print("⚠️ Đang khởi tạo, hãy thử chạy lại ô này sau 5 giây.")
print("=" * 65)


## 🎓 BƯỚC 5: Hướng Dẫn Tự Huấn Luyện (Training / Fine-Tuning) Model

Nếu bạn muốn AI học thêm các dáng người Việt Nam hoặc các bộ sưu tập thời trang riêng:
1. **Cấu trúc thư mục Dataset:**
   ```
   my_dataset/
   ├── cloth/          # Ảnh phẳng (flat-lay) của từng chiếc áo/quần
   ├── image/          # Ảnh người mẫu đang mặc bộ đồ đó
   └── agnostic/       # Ảnh người mẫu đã bôi mờ (mask) vùng áo
   ```
2. **Chạy script huấn luyện:** Chạy ô code bên dưới để bắt đầu quá trình training LoRA trọng số riêng.

In [ ]:
# =============================================================================
# SCRIPT THAM KHẢO HUẤN LUYỆN (TRAINING / FINE-TUNING) THỜI TRANG RIÊNG
# (Lưu ý: Chỉ chạy khi bạn đã chuẩn bị sẵn folder my_dataset/ và muốn train thêm)
# =============================================================================

# Lệnh mẫu (Bỏ dấu # ở đầu dòng để chạy khi đã có dataset):
# !accelerate launch train.py \
#     --pretrained_model_name_or_path="booksforear/sd-image-inpainting" \
#     --train_data_dir="./my_dataset" \
#     --output_dir="./my_finetuned_vton" \
#     --train_batch_size=4 \
#     --gradient_accumulation_steps=2 \
#     --learning_rate=1e-5 \
#     --max_train_steps=2000 \
#     --mixed_precision="fp16"

print("💡 Để chạy máy chủ thử đồ, bạn chỉ cần chạy Bước 1 -> Bước 4 ở trên.")
print("📖 Hướng dẫn training chi tiết: https://github.com/Zheng-Chong/CatVTON")
